# tp-11 購入額の段階づけ（見送り / 500 円 / 1000 円 / 2000 円）

**目的**: 1 レースの購入額を、期待値・的中確率・直前のオッズ低下への強さに応じて 4 段階から選ぶ。
1. **見送り**: 期待値が取れない、または取れても的中確率が異常に低い
2. **500 円**: 期待値がやや高い。直前のオッズ低下を考えてリスクヘッジ
3. **1000 円**: 期待値が高い。オッズが下がってもトントンの範囲
4. **2000 円**: 期待値が十分で、的中確率も悪くない。オッズが下がっても期待値プラスを保てる

## 方法
- レース内の買い目の配分は、tp-08/09 の方策（RL）が決める。段階づけはその上の層で、**レース単位の指標**から購入額を決める。
- 指標: **期待回収率**（配分で重みづけした p × オッズ = 生、等張回帰で実績に較正した値 = cal）、**的中確率**（買い目のどれか 1 つが当たる確率。上位 3 着の並びの全列挙で厳密に計算）、元返し以上になる確率。**オッズ低下の許容率** δ* = 1 − 1/EV（EV × (1 − δ) ≥ 1 を保てる最大の δ）。
- 規則: (a) 分位点の規則（見送り 40% / 500 円 30% / 1000 円 20% / 2000 円 10% に固定）、(b) 絶対しきい値の規則（較正 EV ≥ 1.00 / 1.05 / 1.10。ユーザー案の意味に近い）、(c) 学習年の利益が最大になる規則（探索）。基準 = 毎レース 1000 円 / 2000 円。
- 金額は 100 円単位に按分（最大剰余法）。精算は確定の払戻。
- **段階が上がるほど、実際の回収率が上がるか（EV の指標が実績を分けているか）**を最初に確認する。上がらなければ、段階づけは意味を持たない。
- 頑健性: 判断に使うオッズが確定より δ_true だけ高かった（直前に低下した）と仮定し（0 / 5 / 10 / 20%）、規則は変えずに回収率がどう落ちるかを見る。保守版（EV を 10% 割り引いて判定）も比べる。

fold A = 学習 2022 → 評価 2023、fold B = 学習 2022〜2023 → 評価 2024。最終 = 学習 2022〜2024 → test 2025（一度だけ）。確定オッズでの検証であり、購入時点のオッズではない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time, gc, pickle
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.betting import market_analysis as ma
from src.pipeline.betting import policy as pol
from src.pipeline.betting import staking as sk
from src.pipeline.betting import ticket_env as te
from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP11_TAG", "cmp_v2")                  # tp-09 が作った買い目の表（取消馬を除いた版）を使う
SMOKE = os.environ.get("TP11_SMOKE") == "1"
OUT = BET / "staking" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "staking" / f"TEST2025_EVALUATED_{TAG}.json"
THREADS = max(1, min(4, detect_compute(reserve_cores=6).threads))
SEEDS = (0,) if SMOKE else (0, 1, 2)
FINAL_SEEDS = (0, 1) if SMOKE else (0, 1, 2, 3, 4)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-staking")
require_tracking_server()
YEARS = ["2022", "2023", "2024", "2025"]
TDIR = BET / "tickets" / f"{TAG}_K12_v2"
want = json.loads((TDIR / "2022.json").read_text())
assert want.get("clean") == 1, "取消馬を除いた買い目の表がありません（tp-09 を先に実行）"
A_B, B_B = want["a"], want["b"]
TICK = {y: pd.read_parquet(TDIR / f"{y}.parquet") for y in YEARS}
BEST8 = json.loads((BET / "rl/ens_v2/best_config.json").read_text(encoding="utf-8"))["params"]
wp = pd.read_parquet(BET / "odds_win_place.parquet"); wp = wp[wp["race_id"].str[:4].isin(YEARS) & (wp["win_odds"] > 0)].reset_index(drop=True)
cols = ["race_id", "horse_id", "year", "p_win", "p_top2", "p_top3"]
pred_all = pd.concat([pd.read_parquet(WORK / "oof/model_oof/ensemble_v2.parquet")[cols], pd.read_parquet(WORK / "final_eval_v2/test2025_predictions.parquet")[cols]], ignore_index=True)
pred_all = pred_all[pred_all["year"].between(2022, 2025)]
print("Benter (a, b) =", A_B, B_B, "| 買い目の表:", {y: len(v) for y, v in TICK.items()})

In [ ]:
# レースごとの確率分布（的中確率の厳密計算用）。年ごとにキャッシュ
DIST = {}
DD = BET / "dists" / TAG
DD.mkdir(parents=True, exist_ok=True)
for y in YEARS:
    f = DD / f"{y}_clean.pkl"
    if f.exists():
        DIST[y] = pickle.load(open(f, "rb")); continue
    t0 = time.time()
    ids = set(pred_all.loc[pred_all["year"] == int(y), "race_id"])
    DIST[y] = te.race_dists_parallel(pred_all[pred_all["race_id"].isin(ids)], wp[wp["race_id"].isin(ids)], A_B, B_B, procs=THREADS)
    pickle.dump(DIST[y], open(f, "wb"))
    print(y, len(DIST[y]), f"{time.time() - t0:.0f}s", flush=True)

In [ ]:
def add_cal(dtr):
    xtr = np.clip(dtr["p_comb"].to_numpy() * dtr["odds"].to_numpy(), 0, 5)
    half = (pd.util.hash_array(dtr["race_id"].to_numpy()) % 2).astype(int)
    cal = np.zeros(len(dtr))
    for h in (0, 1):
        iso = ma.IsotonicEV().fit(dtr[half != h], xtr[half != h])
        cal[half == h] = iso.predict(dtr[half == h], xtr[half == h])
    return dtr.assign(cal_ev=cal), ma.IsotonicEV().fit(dtr, xtr)


def run_period(tr_years, te_year, seeds):
    """学習年で方策と較正を当てはめ、評価年の買い目の配分 w と較正 EV を返す。評価年のオッズに係数（判断オッズの上振れ）をかけた版も作れるよう、モデルを返す。"""
    dtr = pd.concat([TICK[y] for y in tr_years], ignore_index=True)
    dtr, iso = add_cal(dtr)
    pk_tr, mu, sd = pol.pack(dtr)
    nets = []
    for s in seeds:
        c = pol.TrainConfig(**{**BEST8, "seed": s, "threads": THREADS})
        nets.append((pol.train_policy(pk_tr, c)[0], c))
    return dtr, iso, (mu, sd), nets


def decide_inputs(dte, iso, musd, nets, factor=1.0):
    """評価年の買い目の表（判断オッズ = 確定 × factor）に、方策の配分 w と較正 EV を付ける。"""
    d = dte.copy().reset_index(drop=True)
    d["odds"] = d["odds"] * factor
    x = np.clip(d["p_comb"].to_numpy() * d["odds"].to_numpy(), 0, 5)
    d["cal_ev"] = iso.predict(d, x)
    pk, _, _ = pol.pack(d, *musd)
    w = np.mean([pol.predict_alloc(n_, pk, c_.max_ticket) for n_, c_ in nets], axis=0)
    ww = np.zeros(len(d))
    m = pk.index >= 0
    ww[pk.index[m]] = w[m]
    d["w"] = ww
    return d

## 1. 段階が上がるほど実績が上がるか（fold A・B）

In [ ]:
FOLDS = [("A", ["2022"], "2023"), ("B", ["2022", "2023"], "2024")]
RESULTS = {}
t0 = time.time()
for fold, tr_y, va_y in FOLDS:
    dtr, iso, musd, nets = run_period(tr_y, va_y, SEEDS)
    # 学習側の指標（較正 EV はクロスフィット済み）。学習側の配分は、評価側と同じ方策を当てはめる（学習データに対する過適合の配分なので、しきい値は評価年の配分で見るのが本来。ここでは学習年の配分で近似）
    dtr_in = dtr.copy().reset_index(drop=True)
    pk_tr, _, _ = pol.pack(dtr_in, *musd)
    w_tr = np.mean([pol.predict_alloc(n_, pk_tr, c_.max_ticket) for n_, c_ in nets], axis=0)
    ww = np.zeros(len(dtr_in)); mk = pk_tr.index >= 0; ww[pk_tr.index[mk]] = w_tr[mk]; dtr_in["w"] = ww
    dist_tr = {k: v for y in tr_y for k, v in DIST[y].items()}
    f_tr = sk.race_features(dtr_in, dist_tr)
    RESULTS[fold] = dict(tr_years=tr_y, va_year=va_y, iso=iso, musd=musd, nets=nets, f_tr=f_tr, dtr=dtr_in)
    print(fold, "学習側のレース", len(f_tr), f"{time.time() - t0:.0f}s", flush=True)
    gc.collect()

In [ ]:
_INP = {}


def evaluate_rules(R, factor=1.0, haircut=0.0):
    """評価年の指標を作り、規則ごとに精算する。factor = 判断オッズの上振れ（確定 × factor）、haircut = EV を割り引いてから判定する率。"""
    va = R["va_year"]
    k_ = (va, tuple(R["tr_years"]), round(factor, 4))
    if k_ not in _INP:
        dte_ = decide_inputs(TICK[va], R["iso"], R["musd"], R["nets"], factor)
        _INP[k_] = (dte_, sk.race_features(dte_, DIST[va]))
    dte, f = _INP[k_][0], _INP[k_][1].copy()
    for c in ("ev_raw", "ev_cal"):
        f[c] = f[c] * (1.0 - haircut)
    f["ev_mix"] = f["ev_cal"] + 1e-3 * f["ev_raw"]
    f_tr = R["f_tr"]
    rules = {
        "分位点（40/30/20/10%）": sk.quantile_rule(f_tr),
        "絶対しきい値（較正EV 1.00/1.05/1.10）": sk.TierRule(1.00, 1.05, 1.10),
    }
    rows, tiers = [], {}
    base = {"毎レース1000円": np.full(len(f), 1000), "毎レース2000円": np.full(len(f), 2000)}
    allst = {**base, **{k: r.assign(f) for k, r in rules.items()}}
    for name, st_ in allst.items():
        sett = sk.settle_tiered(dte, f, st_)
        stake, ret = float(sett["stake"].sum()), float(sett["ret"].sum())
        rows.append({"規則": name, "参加率": float((st_ > 0).mean()), "賭け金(円)": stake, "払戻(円)": ret, "利益(円)": ret - stake, "回収率": ret / stake if stake else np.nan, "レース数": len(f)})
        tiers[name] = sk.summarize_tiers(f, st_, sett)
    return pd.DataFrame(rows), tiers, f, dte, rules


def decile_table(R):
    """指標の十分位別に、毎レース 1000 円で精算した回収率。指標が実績を分けていなければ、段階づけに意味がない。"""
    summ, tiers, f_va, dte, rules = evaluate_rules(R)
    sett = sk.settle_tiered(dte, f_va, np.full(len(f_va), 1000))
    m = f_va.merge(sett, on="race_id", how="left")
    rows = []
    for col in ("ev_raw", "ev_cal", "hit", "ppos"):
        m["d"] = pd.qcut(m[col].rank(method="first"), 10, labels=False)
        g = m.groupby("d").agg(races=("race_id", "size"), stake=("stake", "sum"), ret=("ret", "sum"), 指標=(col, "mean"))
        g["回収率"] = g["ret"] / g["stake"]
        g.insert(0, "指標名", col)
        rows.append(g.reset_index())
    return pd.concat(rows, ignore_index=True)


TAB = {}
DEC = {}
for fold, tr_y, va_y in FOLDS:
    DEC[fold] = decile_table(RESULTS[fold])
    print(f"== fold {fold}（評価 {va_y}）: 指標の十分位別の回収率 =="); display(DEC[fold].pivot_table(index="d", columns="指標名", values="回収率").round(3))
    DEC[fold].to_csv(OUT / f"deciles_{fold}.csv", index=False)
    summ, tiers, f_va, dte, rules = evaluate_rules(RESULTS[fold])
    TAB[fold] = (summ, tiers)
    print(f"== fold {fold}（評価 {va_y}）=="); display(summ.round(3).set_index("規則"))
    for k in ("分位点（40/30/20/10%）", "絶対しきい値（較正EV 1.00/1.05/1.10）"):
        print(k, "→ 段階別"); display(tiers[k].round(3))

## 2. 規則の探索（学習年の利益が最大になるしきい値）と、オッズ低下への頑健性
- 探索: 見送り・500・1000・2000 円の割合を (見送り 20〜80%) の格子から選び、学習側の利益が最大のもの。優位が無ければ「全部見送り」になる（その場合は、参加率の下限 25% を課した版も出す）。
- 頑健性: 評価年の判断オッズを確定 × (1 + δ) に上振れさせ（直前にオッズが下がる状況の模擬）、規則・方策は変えずに精算。

In [ ]:
def search_rule(R, min_part=0.0):
    """学習側で、分位点の組を格子探索して利益（円）を最大にする規則。"""
    f_tr, dtr = R["f_tr"], R["dtr"]
    best, best_p = None, -1e18
    for skip in (0.2, 0.4, 0.6, 0.8):
        for s2, s3 in ((0.4, 0.4), (0.5, 0.3), (0.3, 0.5), (0.6, 0.2)):
            rest = 1.0 - skip
            sh = (skip, rest * s2, rest * s3 * 0.7, rest * s3 * 0.3 + rest * (1 - s2 - s3))
            sh = tuple(np.array(sh) / np.sum(sh))
            if 1 - sh[0] < min_part:
                continue
            rule = sk.quantile_rule(f_tr, shares=sh)
            st_ = rule.assign(f_tr)
            sett = sk.settle_tiered(dtr, f_tr, st_)
            p = float(sett["profit"].sum())
            if p > best_p:
                best, best_p = (rule, sh), p
    return best


rows = []
for fold, tr_y, va_y in FOLDS:
    R = RESULTS[fold]
    searched = {"利益最大": search_rule(R), "利益最大（参加25%以上）": search_rule(R, 0.25)}
    for dl in (0.0, 0.05, 0.10, 0.20):
        for hc_name, hc in (("通常", 0.0), ("保守（EV10%割引）", 0.10)):
            summ, tiers, f_va, dte, rules = evaluate_rules(R, factor=1.0 + dl, haircut=hc)
            for k, (rule, sh) in searched.items():
                st_ = rule.assign(f_va)
                sett = sk.settle_tiered(dte, f_va, st_)
                stake, ret = float(sett["stake"].sum()), float(sett["ret"].sum())
                summ = pd.concat([summ, pd.DataFrame([{"規則": f"探索: {k}", "参加率": float((st_ > 0).mean()), "賭け金(円)": stake, "払戻(円)": ret, "利益(円)": ret - stake, "回収率": ret / stake if stake else np.nan, "レース数": len(f_va)}])], ignore_index=True)
            summ.insert(0, "fold", fold); summ.insert(1, "判断オッズの上振れδ", dl); summ.insert(2, "判定", hc_name)
            rows.append(summ)
ST = pd.concat(rows, ignore_index=True)
ST.to_csv(OUT / "valid_stress.csv", index=False)
piv = ST.pivot_table(index=["規則", "判定"], columns=["fold", "判断オッズの上振れδ"], values="回収率")
display(piv.round(3))

## 3. 最終: 学習 2022〜2024 → test 2025（一度だけ）
fold で見た手順をそのまま、2025 年に適用する。

In [ ]:
if not SMOKE and LOCK.exists() and os.environ.get("TP11_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）。結果を見てやり直さないため、再実行しません。")
TEST_Y = "2024" if SMOKE else "2025"
TR_Y = ["2022", "2023"] if SMOKE else ["2022", "2023", "2024"]
dtr, iso, musd, nets = run_period(TR_Y, TEST_Y, FINAL_SEEDS)
dtr_in = dtr.copy().reset_index(drop=True)
pk_tr, _, _ = pol.pack(dtr_in, *musd)
w_tr = np.mean([pol.predict_alloc(n_, pk_tr, c_.max_ticket) for n_, c_ in nets], axis=0)
ww = np.zeros(len(dtr_in)); mk = pk_tr.index >= 0; ww[pk_tr.index[mk]] = w_tr[mk]; dtr_in["w"] = ww
FR = dict(tr_years=TR_Y, va_year=TEST_Y, iso=iso, musd=musd, nets=nets, f_tr=sk.race_features(dtr_in, {k: v for y in TR_Y for k, v in DIST[y].items()}), dtr=dtr_in)
searched = {"利益最大": search_rule(FR), "利益最大（参加25%以上）": search_rule(FR, 0.25)}
rows, tier_tabs = [], {}
for dl in (0.0, 0.05, 0.10, 0.20):
    for hc_name, hc in (("通常", 0.0), ("保守（EV10%割引）", 0.10)):
        summ, tiers, f_te, dte, rules = evaluate_rules(FR, factor=1.0 + dl, haircut=hc)
        for k, (rule, sh) in searched.items():
            st_ = rule.assign(f_te); sett = sk.settle_tiered(dte, f_te, st_)
            stake, ret = float(sett["stake"].sum()), float(sett["ret"].sum())
            summ = pd.concat([summ, pd.DataFrame([{"規則": f"探索: {k}", "参加率": float((st_ > 0).mean()), "賭け金(円)": stake, "払戻(円)": ret, "利益(円)": ret - stake, "回収率": ret / stake if stake else np.nan, "レース数": len(f_te)}])], ignore_index=True)
        summ.insert(0, "判断オッズの上振れδ", dl); summ.insert(1, "判定", hc_name)
        rows.append(summ)
        if dl == 0.0 and hc == 0.0:
            tier_tabs = tiers
FIN = pd.concat(rows, ignore_index=True)
FIN.to_csv(OUT / "final_test.csv", index=False)
display(FIN.pivot_table(index=["規則", "判定"], columns="判断オッズの上振れδ", values="回収率").round(3))
display(FIN[(FIN["判断オッズの上振れδ"] == 0.0) & (FIN["判定"] == "通常")].set_index("規則").round(3))
for k, t in tier_tabs.items():
    print(k, "→ 段階別（δ=0・通常）"); display(t.round(3)); t.to_csv(OUT / f"final_tiers__{k[:6]}.csv")

In [ ]:
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "test_year": TEST_Y, "smoke": SMOKE,
       "searched_rules": {k: {"shares": list(map(float, sh)), "rule": r.describe()} for k, (r, sh) in searched.items()},
       "final": FIN.to_dict("records"), "valid": ST.to_dict("records"),
       "notes": ["確定オッズでの検証（購入時点のオッズではない）", "判断オッズの上振れδは、直前のオッズ低下の粗い模擬"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
if not SMOKE:
    with mlflow.start_run(run_name=f"tp-11-staking-{TAG}", tags={"notebook": "tp-11", "git_commit": GIT_COMMIT, "test_year": "2025"}):
        for i_, (_, r) in enumerate(FIN[(FIN["判断オッズの上振れδ"] == 0.0) & (FIN["判定"] == "通常")].iterrows()):
            mlflow.log_metrics({f"test2025_roi__{i_:02d}": float(r["回収率"]) if pd.notna(r["回収率"]) else 0.0})
        mlflow.log_dict({f"{i_:02d}": r["規則"] for i_, (_, r) in enumerate(FIN[(FIN["判断オッズの上振れδ"] == 0.0) & (FIN["判定"] == "通常")].iterrows())}, "staking/metric_index.json")
        for p in OUT.glob("*"):
            mlflow.log_artifact(str(p), "staking")
    LOCK.write_text(json.dumps({"evaluated_at": res["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 の段階づけ評価は実施済み。結果を見てやり直さない。"}, ensure_ascii=False))
print("完了")